# Demo 2: Recursive Character Splitting & Overlap

## 1. Theory & Concepts
When we have a long parsed clean text (e.g., Item 7 is 10,000 words long), we cannot load it all directly into a vector database or an LLM. We must perform **Chunking (Splitting)**.

### Recursive Character Splitting Algorithm:
This is the standard splitting algorithm for prose-style documents. Instead of rigidly cutting text at arbitrary positions (which easily splits a sentence or word in half), this algorithm uses a list of separators with decreasing priority:
1. `\n\n` (Paragraph break - Highest priority to keep paragraphs intact)
2. `\n` (Line break)
3. ` ` (Space - Separating words)
4. `""` (Empty string - Splits character by character, a fallback solution)

**Recursive Mechanism:** The algorithm first attempts to split the text using the `\n\n` separator. If the resulting chunks are smaller than `chunk_size`, they are kept. If any chunk is larger than `chunk_size`, the algorithm recursively calls itself on that specific chunk using the next separator in the list (the `\n` separator), followed by space, and so on, until all chunks are within the limit.

### Concept of Overlap:
When splitting text, context near the boundary of the split can easily be lost (e.g., the preceding sentence goes to Chunk 1, while the following sentence is pushed to Chunk 2). 
To solve this, **Chunk Overlap** defines a region of overlapping text. The end of Chunk 1 is repeated at the beginning of Chunk 2. This maintains **Context Continuity**.

### Data Flow:
* **Input:** Clean text string from the Parsing step (e.g., clean text of Item 7).
* **Output:** A list of small text segments (chunks) of a specified maximum length, along with full metadata.
* **Purpose of Output:** These chunks will be written to the final storage file (`data/processed/documents.jsonl`). Subsequently, the indexing scripts (BM25, FAISS Vector) will read this JSONL file to perform embedding and build the search indexes.

## 2. Initialize Sample Text (Input)
We prepare a sample text about Amazon's business performance to perform chunking.

In [ ]:
sample_text = (
    "We are Amazon.com, Inc. and we focus on customer obsession. During 2023, our net sales "
    "increased 12% to $574.8 billion compared to $514.0 billion in 2022. This performance was "
    "primarily driven by AWS cloud segment growth and net sales in retail.\n\n"
    "Our operating income increased to $36.9 billion in 2023, compared to $12.2 billion in 2022. "
    "We expect the macroeconomic environment to continue to impact our operational results. "
    "Therefore, we make significant investments in our software and satellite networks to secure "
    "our long-term growth opportunities.\n\n"
    "Our capital expenditures (specifically purchases of property and equipment) were $52,729 million "
    "in fiscal year 2023, down from $63,645 million in fiscal year 2022."
)
print("=== INPUT (Text to Chunk) ===")
print(sample_text)

## 3. Implementing the Recursive Splitting Algorithm Simulation
We will write the recursive splitting algorithm from scratch in Python to print detailed step-by-step logs of its execution.

In [ ]:
def recursive_split(text, separators, max_size, overlap):
    """
    Recursive splitting function simulating LangChain's logic
    """
    # If text is small enough, return immediately
    if len(text) <= max_size:
        return [text]
    
    # If no separators left, force hard split by length
    if not separators:
        return [text[i:i+max_size] for i in range(0, len(text), max_size - overlap)]
    
    # Get current highest priority separator
    separator = separators[0]
    next_separators = separators[1:]
    
    # Split text into smaller segments
    print(f"[Log] Trying separator: '{repr(separator)}'")
    splits = text.split(separator)
    
    chunks = []
    current_chunk = ""
    
    for split in splits:
        # If split segment is too large, recursively split using next separator
        if len(split) > max_size:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""
            
            print(f"  -> Detected long segment ({len(split)} characters) exceeding {max_size}. Recursively splitting...")
            recursive_chunks = recursive_split(split, next_separators, max_size, overlap)
            chunks.extend(recursive_chunks)
        else:
            # Merge small splits until close to max_size limit
            potential_chunk = current_chunk + (separator if current_chunk else "") + split
            if len(potential_chunk) <= max_size:
                current_chunk = potential_chunk
            else:
                # Limit reached, save current chunk
                if current_chunk:
                    chunks.append(current_chunk)
                # Initialize new chunk, preserving Overlap from end of previous chunk
                overlap_text = current_chunk[-overlap:] if current_chunk and overlap > 0 else ""
                current_chunk = overlap_text + (separator if overlap_text else "") + split
                
    if current_chunk:
        chunks.append(current_chunk)
        
    return chunks

# Set chunking parameters
CHUNK_SIZE = 250
CHUNK_OVERLAP = 50
SEPARATORS = ["\n\n", "\n", " ", ""]

print(f"=== CHUNKING CONFIGURATION ===")
print(f"Chunk Size (Max length): {CHUNK_SIZE} characters")
print(f"Chunk Overlap (Overlap width): {CHUNK_OVERLAP} characters\n")

final_chunks = recursive_split(sample_text, SEPARATORS, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"\n[SUCCESS] Split into {len(final_chunks)} chunks!")

## 4. Visualizing Chunking Results and Overlap
We display each chunk in its entirety to observe the split boundaries and the overlapping text.

In [ ]:
print("=== CHUNKING OUTPUT RESULTS ===")
for idx, chunk in enumerate(final_chunks):
    print(f"\n----------------- CHUNK {idx+1} (Length: {len(chunk)} characters) -----------------")
    print(chunk)
    
# Visualize Overlap between Chunk 1 and Chunk 2
if len(final_chunks) >= 2:
    c1 = final_chunks[0]
    c2 = final_chunks[1]
    
    # Find matching substring at end of c1 and start of c2
    overlap_found = ""
    for i in range(1, len(c2)):
        suffix = c1[-i:]
        if c2.startswith(suffix):
            overlap_found = suffix
            
    print("\n" + "="*60)
    print("VISUALIZING OVERLAP (CONTEXT INTERSECTION)")
    print("="*60)
    print(f"Overlap found at end of Chunk 1 and start of Chunk 2 (Length: {len(overlap_found)} characters):")
    print(f"👉 \"{overlap_found}\"")
    print("="*60)

## 5. Conclusion and Final Output
Once chunking is complete, each chunk will be packaged with its metadata (Ticker, Year, Section, Page, etc.).
This output data will be written to the `data/processed/documents.jsonl` file in JSON Lines format. This structure ensures a unified storage format across the system, making it easily scalable and ready for the next stage of **Indexing**.